# Marker Gene Selection

In [ ]:
import pandas as pd
import numpy as np
from utils.misc import extract_number

In [ ]:
# load data
gene_expressions = pd.read_csv("data/train_data.csv", index_col=0)
gene_expressions_mat = gene_expressions.to_numpy()
genenames = np.array(gene_expressions.index.tolist())
samples = gene_expressions.columns.tolist()

# extract ages
ages = np.array([extract_number(timestring) for timestring in samples])
unique_ages=np.unique(ages)

# retain genes that are present in all samples
prevalence = np.mean(gene_expressions_mat > 0, axis=1)
subset_gene_id = np.where(prevalence == 1)[0]
subset_genenames = genenames[subset_gene_id]
gene_expressions = gene_expressions.loc[subset_genenames, :]
gene_expressions_mat = gene_expressions_mat[subset_gene_id, :]

# get log expressions
log_gene_expressions = np.log(gene_expressions)
log_gene_expressions_mat = np.log(gene_expressions_mat)

# transpose count tables to samples by genes
gene_expressions = gene_expressions.T
gene_expressions_mat = gene_expressions_mat.T
log_gene_expressions = log_gene_expressions.T
log_gene_expressions_mat = log_gene_expressions_mat.T

# get rankings of samples for each gene expression
gene_expressions_rank = log_gene_expressions.rank()

In [ ]:
nsamples = len(ages)

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests
from utils.variation import var_comp, f_test_filter, hcluster_genes
from scipy.stats import f, t
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster
from utils.viz import single_scatter_plot, save_geneexp_scatter_plots

## Find genes that have strong variation between age 6 and 23
I first calculate F statistic for each gene to filter out genes that do not differ between samples of different ages between age 6 and 23. For the rest of the genes, I calculate the mean expression at each age, normalize them and apply hierarchical clustering to all the genes.

In [ ]:
subset_genes, variance_df_subset = f_test_filter(geneexp_df=gene_expressions,
                                                ages=ages,
                                                min_age=6, max_age=23)

In [ ]:
Z = hcluster_genes(geneexp_df=gene_expressions, ages=ages,
                            subset_genes=subset_genes, min_age=6, max_age=23)

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Genes")
plt.ylabel("Distance")
plt.show()
# plt.savefig("gene_plots/markergene_6_23_hclust.pdf", format="pdf", bbox_inches="tight")

Based on the dendrogram above, I decide to split the genes into 10 clusters.

In [ ]:
clusters = fcluster(Z, t=10, criterion='maxclust')
np.unique(clusters, return_counts=True)

In [ ]:
variance_df_subset["Cluster"] = clusters

In [ ]:
marker_genes_6_23 = []
for cnumber in np.arange(1, 11):
    selected_clusters = variance_df_subset.loc[variance_df_subset["Cluster"] == cnumber, :].sort_values(by="P_Value")
    cluster_genes = selected_clusters.index.tolist()
    marker_genes_6_23.extend(cluster_genes[0:int(len(cluster_genes)/5)])

In [ ]:
variance_df_subset.loc[marker_genes_6_23, :].to_csv("gene_plots/top_gene_6_23/markergene_6_23.csv")

Save plots all the marker genes.

In [ ]:
save_geneexp_scatter_plots(geneexp_df=gene_expressions, variance_df=variance_df_subset, marker_genes=marker_genes_6_23,
                   ages=ages, folder="gene_plots/top_gene_6_23")

## Find genes that have strong variation between age 2 and 23

In [ ]:
subset_genes, variance_df_subset = f_test_filter(geneexp_df=gene_expressions,
                                                ages=ages,
                                                min_age=2, max_age=23)

In [ ]:
Z = hcluster_genes(geneexp_df=gene_expressions, ages=ages,
                            subset_genes=subset_genes, min_age=2, max_age=23)

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Genes")
plt.ylabel("Distance")
# plt.show()
plt.savefig("gene_plots/top_gene_2_23/markergene_2_23_hclust.pdf", format="pdf", bbox_inches="tight")

I decide to cut the dendrogram at 15 clusters, and pick 1/40 top genes from each cluster

In [ ]:
clusters = fcluster(Z, t=15, criterion='maxclust')
np.unique(clusters, return_counts=True)

In [ ]:
variance_df_subset["Cluster"] = clusters

In [ ]:
marker_genes_2_23 = []
for cnumber in np.arange(1, 16):
    selected_clusters = variance_df_subset.loc[variance_df_subset["Cluster"] == cnumber, :].sort_values(by="P_Value")
    cluster_genes = selected_clusters.index.tolist()
    marker_genes_2_23.extend(cluster_genes[0:int(len(cluster_genes)/40)])

In [ ]:
variance_df_subset.loc[marker_genes_2_23, :].to_csv("gene_plots/top_gene_2_23/markergene_2_23.csv")

In [ ]:
save_geneexp_scatter_plots(geneexp_df=gene_expressions, variance_df=variance_df_subset, marker_genes=marker_genes_2_23,
                   ages=ages, folder="gene_plots/top_gene_2_23")

## Find genes that have strong variation between age 2 and 18

In [ ]:
subset_genes, variance_df_subset = f_test_filter(geneexp_df=gene_expressions,
                                                ages=ages,
                                                min_age=2, max_age=18)

In [ ]:
Z = hcluster_genes(geneexp_df=gene_expressions, ages=ages,
                            subset_genes=subset_genes, min_age=2, max_age=18)

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Genes")
plt.ylabel("Distance")
# plt.show()
plt.savefig("gene_plots/top_gene_2_18/markergene_2_18_hclust.pdf", format="pdf", bbox_inches="tight")

In [ ]:
clusters = fcluster(Z, t=20, criterion='maxclust')
np.unique(clusters, return_counts=True)

In [ ]:
variance_df_subset["Cluster"] = clusters

In [ ]:
marker_genes_2_18 = []
for cnumber in np.arange(1, 21):
    selected_clusters = variance_df_subset.loc[variance_df_subset["Cluster"] == cnumber, :].sort_values(by="P_Value")
    cluster_genes = selected_clusters.index.tolist()
    marker_genes_2_18.extend(cluster_genes[0:int(len(cluster_genes)/40)])

In [ ]:
variance_df_subset.loc[marker_genes_2_18, :].to_csv("gene_plots/top_gene_2_18/markergene_2_18.csv")

In [ ]:
save_geneexp_scatter_plots(geneexp_df=gene_expressions, variance_df=variance_df_subset, marker_genes=marker_genes_2_18,
                   ages=ages, folder="gene_plots/top_gene_2_18")

## Find genes that have strong variation between age 6 and 18

In [ ]:
subset_genes, variance_df_subset = f_test_filter(geneexp_df=gene_expressions,
                                                ages=ages,
                                                min_age=6, max_age=18)

In [ ]:
Z = hcluster_genes(geneexp_df=gene_expressions, ages=ages,
                            subset_genes=subset_genes, min_age=6, max_age=18)

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Genes")
plt.ylabel("Distance")
# plt.show()
plt.savefig("gene_plots/top_gene_6_18/markergene_6_18_hclust.pdf", format="pdf", bbox_inches="tight")

In [ ]:
clusters = fcluster(Z, t=15, criterion='maxclust')
np.unique(clusters, return_counts=True)

In [ ]:
variance_df_subset["Cluster"] = clusters

In [ ]:
marker_genes_6_18 = []
for cnumber in np.arange(1, 16):
    selected_clusters = variance_df_subset.loc[variance_df_subset["Cluster"] == cnumber, :].sort_values(by="P_Value")
    cluster_genes = selected_clusters.index.tolist()
    marker_genes_6_18.extend(cluster_genes[0:int(len(cluster_genes)/1)])

In [ ]:
variance_df_subset.loc[marker_genes_6_18, :].to_csv("gene_plots/top_gene_6_18/markergene_6_18.csv")

In [ ]:
save_geneexp_scatter_plots(geneexp_df=gene_expressions, variance_df=variance_df_subset, marker_genes=marker_genes_6_18,
                   ages=ages, folder="gene_plots/top_gene_6_18")